## 1. Audit et nettoyage

In [1]:
import pandas as pd

df = pd.read_csv("../data/textes_films_horreur_comedie.csv")

df.head()

,titre,resume,cible
0,Chiedo asilo,Roberto has a new job as the teacher at a nurs...,0
1,Friday the 13th: A New Beginning,Five years after killing the goalie hockey-mas...,1
2,Stranger Than Fiction,Everybody knows that your life is a story. But...,0
3,Paris,"Pierre, a professional dancer, suffers from a ...",0
4,Sleepless,A retired detective with insomnia is called up...,1


In [2]:
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))

Nombre de lignes : 2400
Nombre de colonnes : 3


In [3]:
df["cible"].value_counts()

cible
0    1200
1    1200
Name: count, dtype: int64

In [4]:
df["longueur_resume"] = df["resume"].str.len()

df["longueur_resume"].describe().round(2)

count    2400.00
mean      558.19
std       349.28
min       201.00
25%       335.00
50%       474.00
75%       679.00
max      7104.00
Name: longueur_resume, dtype: float64

In [5]:
print("Doublons avant nettoyage :", df.duplicated().sum())

df["titre"] = df["titre"].str.strip()
df["resume"] = df["resume"].str.strip()

df = df.drop_duplicates(subset=["titre", "resume", "cible"]).copy()

print("Doublons après nettoyage :", df.duplicated().sum())

Doublons avant nettoyage : 54
Doublons après nettoyage : 0


In [6]:
print(df.isnull().sum())

titre              0
resume             0
cible              0
longueur_resume    0
dtype: int64


In [7]:
print("Valeurs nulles restantes :")
print(df[["resume", "cible"]].isnull().sum())

print("\nDoublons restants :", df.duplicated(subset=["resume"]).sum())

print("\nNombre de lignes finales :", len(df))

Valeurs nulles restantes :
resume    0
cible     0
dtype: int64

Doublons restants : 0

Nombre de lignes finales : 2346


## 2. Découpage du jeu de données

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
df_train, df_temp = train_test_split(
    df,
    test_size=0.40,
    random_state=42,
    stratify=df["cible"]
)

In [10]:
df_test, df_prod = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["cible"]
)

In [11]:
print("Apprentissage :", len(df_train))
print("Test :", len(df_test))
print("Production :", len(df_prod))

print("\nRépartition apprentissage :")
print(df_train["cible"].value_counts(normalize=True))

print("\nRépartition test :")
print(df_test["cible"].value_counts(normalize=True))

print("\nRépartition production :")
print(df_prod["cible"].value_counts(normalize=True))

Apprentissage : 1407
Test : 469
Production : 470

Répartition apprentissage :
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64

Répartition test :
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64

Répartition production :
cible
0    0.508511
1    0.491489
Name: proportion, dtype: float64


In [12]:
train_textes = set(df_train["resume"])
test_textes = set(df_test["resume"])
prod_textes = set(df_prod["resume"])

print("Doublons train/test :", len(train_textes & test_textes))
print("Doublons train/production :", len(train_textes & prod_textes))
print("Doublons test/production :", len(test_textes & prod_textes))

Doublons train/test : 0
Doublons train/production : 0
Doublons test/production : 0


Le jeu de données nettoyé a été séparé en trois ensembles : 60 % pour l'apprentissage, 20 % pour le test et 20 % pour la production. La stratification permet de conserver une répartition similaire des catégories horreur et comédie. Une vérification des résumés identiques entre les ensembles a également été effectuée afin de prévenir les fuites de données.

## 3. Représentation du texte(TF-IDF)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
tfidf = TfidfVectorizer(
    max_features=5000
)

In [15]:
X_train = tfidf.fit_transform(df_train["resume"])

In [16]:
X_test = tfidf.transform(df_test["resume"])
X_prod = tfidf.transform(df_prod["resume"])

In [17]:
y_train = df_train["cible"]
y_test = df_test["cible"]
y_prod = df_prod["cible"]

In [18]:
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("X_prod :", X_prod.shape)

print("\nNombre de mots dans le vocabulaire :", len(tfidf.vocabulary_))

X_train : (1407, 5000)
X_test : (469, 5000)
X_prod : (470, 5000)

Nombre de mots dans le vocabulaire : 5000


In [19]:
mots = tfidf.get_feature_names_out()

print("Exemples de mots du vocabulaire :")
print(mots[:30])

Exemples de mots du vocabulaire :
['000' '10' '100' '11' '12' '13' '13th' '14' '15' '17' '18' '18th' '1950'
 '1950s' '1955' '1960' '1962' '1968' '1974' '1980s' '1988' '1993' '19th'
 '20' '2011' '20th' '21' '21st' '22' '24']


In [20]:
mots = tfidf.get_feature_names_out()

# Transformation du premier résumé en tableau
vecteur = X_train[0].toarray()[0]

# Association des mots à leurs poids
resultat = pd.DataFrame({
    "mot": mots,
    "poids": vecteur
})

# Afficher les 10 mots les plus importants
resultat = resultat[resultat["poids"] > 0]
resultat.sort_values("poids", ascending=False).head(10)

,mot,poids
1036,cousin,0.291108
957,conquer,0.188301
1496,era,0.188301
4293,subsequent,0.188301
3259,pact,0.182179
4536,to,0.178379
84,acquaintance,0.177177
481,berlin,0.172948
3395,poet,0.172948
2170,illness,0.172948


Les 2 346 résumés nettoyés ont été vectorisés à l'aide de TF-IDF, avec un vocabulaire limité à 5 000 termes et appris uniquement sur les 1 407 exemples d'apprentissage. Les ensembles de test (469 exemples) et de production (470 exemples) ont ensuite été transformés avec le même vectoriseur, sans réapprentissage du vocabulaire, afin d'éviter toute fuite de données.

## 4. Les points de comparaison

In [21]:
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

In [22]:
graines = [42, 21, 123, 456, 789]
resultats = []

In [23]:
def creer_modele(nb_features):
    modele = keras.Sequential([
        layers.Input(shape=(nb_features,)),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    modele.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return modele

In [24]:
for graine in graines:

    print(f"\n===== Graine {graine} =====")

    # Découpage interne
    train_interne, val_interne = train_test_split(
        df_train,
        test_size=0.2,
        random_state=graine,
        stratify=df_train["cible"]
    )

    # TF-IDF appris uniquement sur l'entraînement interne
    vectoriseur = TfidfVectorizer(max_features=5000)

    X_tr = vectoriseur.fit_transform(train_interne["resume"])
    X_val = vectoriseur.transform(val_interne["resume"])

    y_tr = train_interne["cible"].to_numpy()
    y_val = val_interne["cible"].to_numpy()

    # 1. Régression logistique
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_tr, y_tr)

    pred_logreg = logreg.predict(X_val)
    f1_logreg = f1_score(y_val, pred_logreg)

    # 2. Naive Bayes
    nb = MultinomialNB()
    nb.fit(X_tr, y_tr)

    pred_nb = nb.predict(X_val)
    f1_nb = f1_score(y_val, pred_nb)

    # 3. Réseau de neurones
    keras.utils.set_random_seed(graine)

    modele = creer_modele(X_tr.shape[1])

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    modele.fit(
        X_tr.toarray(),
        y_tr,
        validation_data=(X_val.toarray(), y_val),
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    probabilites = modele.predict(
        X_val.toarray(),
        verbose=0
    ).ravel()

    pred_reseau = (probabilites >= 0.5).astype(int)
    f1_reseau = f1_score(y_val, pred_reseau)

    # Enregistrement des résultats
    resultats.append({
        "graine": graine,
        "Logistique": f1_logreg,
        "Naive Bayes": f1_nb,
        "Réseau": f1_reseau
    })

    print(f"Logistique : {f1_logreg:.4f}")
    print(f"Naive Bayes : {f1_nb:.4f}")
    print(f"Réseau : {f1_reseau:.4f}")


===== Graine 42 =====
Logistique : 0.8409
Naive Bayes : 0.8736
Réseau : 0.8604

===== Graine 21 =====
Logistique : 0.8459
Naive Bayes : 0.8582
Réseau : 0.8612

===== Graine 123 =====
Logistique : 0.8102
Naive Bayes : 0.8507
Réseau : 0.8223

===== Graine 456 =====
Logistique : 0.8291
Naive Bayes : 0.8736
Réseau : 0.8511

===== Graine 789 =====
Logistique : 0.8120
Naive Bayes : 0.8529
Réseau : 0.8235


In [25]:
df_resultats = pd.DataFrame(resultats)

display(df_resultats.round(4))

,graine,Logistique,Naive Bayes,Réseau
0,42,0.8409,0.8736,0.8604
1,21,0.8459,0.8582,0.8612
2,123,0.8102,0.8507,0.8223
3,456,0.8291,0.8736,0.8511
4,789,0.8120,0.8529,0.8235


In [26]:
comparaison = df_resultats[
    ["Logistique", "Naive Bayes", "Réseau"]
].agg(["mean", "std"])

display(comparaison.round(4))

,Logistique,Naive Bayes,Réseau
mean,0.8276,0.8618,0.8437
std,0.0163,0.0111,0.0194


In [27]:
# Scores F1 moyens sur les 5 graines
scores_moyens = df_resultats[
    ["Logistique", "Naive Bayes", "Réseau"]
].mean()

# Taille de l'ensemble de validation
n = len(val_interne)

# Calcul des marges de bruit
marges = {}

for nom, p in scores_moyens.items():
    bruit = 1.96 * np.sqrt(p * (1 - p) / n)
    marges[nom] = bruit

    print(f"{nom} : F1 = {p:.4f} | Marge = ±{bruit:.4f}")

# Comparaison des modèles deux à deux
print("\n--- Comparaison des modèles ---")

comparaisons = [
    ("Logistique", "Naive Bayes"),
    ("Logistique", "Réseau"),
    ("Naive Bayes", "Réseau")
]

for modele_a, modele_b in comparaisons:

    ecart = abs(scores_moyens[modele_a] - scores_moyens[modele_b])

    # On retient la marge la plus grande des deux
    marge = max(marges[modele_a], marges[modele_b])

    print(f"\n{modele_a} vs {modele_b}")
    print(f"Écart F1 : {ecart:.4f}")
    print(f"Marge de bruit : {marge:.4f}")

    if ecart > marge:
        print("Écart supérieur à la marge de bruit")
    else:
        print("Écart inférieur ou égal à la marge de bruit")

# Meilleur modèle selon le F1 moyen
meilleur_modele = scores_moyens.idxmax()

print("\n--- Meilleur modèle observé ---")
print(f"Modèle : {meilleur_modele}")
print(f"F1 moyen : {scores_moyens.max():.4f}")

Logistique : F1 = 0.8276 | Marge = ±0.0441
Naive Bayes : F1 = 0.8618 | Marge = ±0.0403
Réseau : F1 = 0.8437 | Marge = ±0.0424

--- Comparaison des modèles ---

Logistique vs Naive Bayes
Écart F1 : 0.0342
Marge de bruit : 0.0441
Écart inférieur ou égal à la marge de bruit

Logistique vs Réseau
Écart F1 : 0.0161
Marge de bruit : 0.0441
Écart inférieur ou égal à la marge de bruit

Naive Bayes vs Réseau
Écart F1 : 0.0181
Marge de bruit : 0.0424
Écart inférieur ou égal à la marge de bruit

--- Meilleur modèle observé ---
Modèle : Naive Bayes
F1 moyen : 0.8618


Naive Bayes présente le meilleur F1 moyen parmi les trois modèles évalués.   
Cependant, les écarts observés restent inférieurs aux marges de bruit indicatives.   
Aucun modèle ne peut donc être déclaré clairement supérieur selon le critère retenu.

## 5. Recherche des réglages avec KerasTuner

In [28]:
import keras_tuner as kt
from tensorflow.keras import regularizers

In [29]:
train_tuning, val_tuning = train_test_split(
    df_train,
    test_size=0.2,
    random_state=42,
    stratify=df_train["cible"]
)

tfidf_tuning = TfidfVectorizer(max_features=5000)

X_tuning = tfidf_tuning.fit_transform(
    train_tuning["resume"]
).toarray()

X_validation = tfidf_tuning.transform(
    val_tuning["resume"]
).toarray()

y_tuning = train_tuning["cible"].to_numpy()
y_validation = val_tuning["cible"].to_numpy()

print("Entraînement :", X_tuning.shape)
print("Validation :", X_validation.shape)

Entraînement : (1125, 5000)
Validation : (282, 5000)


In [30]:
def construire_reseau(
    nb_couches,
    nb_neurones,
    dropout,
    regularisation,
    learning_rate
):
    modele = keras.Sequential()

    modele.add(layers.Input(shape=(X_tuning.shape[1],)))

    for _ in range(nb_couches):
        modele.add(
            layers.Dense(
                nb_neurones,
                activation="relu",
                kernel_regularizer=regularizers.l2(regularisation)
            )
        )

        modele.add(layers.Dropout(dropout))

    modele.add(layers.Dense(1, activation="sigmoid"))

    modele.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return modele

In [31]:
def construire_modele_tuner(hp):

    nb_couches = hp.Choice(
        "nb_couches",
        values=[1, 2, 3]
    )

    nb_neurones = hp.Choice(
        "nb_neurones",
        values=[16, 32, 64, 128]
    )

    dropout = hp.Choice(
        "dropout",
        values=[0.1, 0.2, 0.3, 0.4]
    )

    regularisation = hp.Choice(
        "regularisation",
        values=[0.0001, 0.001, 0.01]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.001, 0.01]
    )

    return construire_reseau(
        nb_couches,
        nb_neurones,
        dropout,
        regularisation,
        learning_rate
    )

In [32]:
keras.utils.set_random_seed(42)

tuner = kt.RandomSearch(
    hypermodel=construire_modele_tuner,
    objective="val_accuracy",
    max_trials=20,
    executions_per_trial=1,
    seed=42,
    directory="tuning",
    project_name="films_keras",
    overwrite=True
)

In [33]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

tuner.search(
    X_tuning,
    y_tuning,
    validation_data=(X_validation, y_validation),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Trial 20 Complete [00h 00m 14s]
val_accuracy: 0.847517728805542

Best val_accuracy So Far: 0.8865247964859009
Total elapsed time: 00h 03m 20s


In [34]:
meilleurs_hp = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print("Meilleurs hyperparamètres :")

for nom, valeur in meilleurs_hp.values.items():
    print(f"{nom} : {valeur}")

Meilleurs hyperparamètres :
nb_couches : 3
nb_neurones : 32
dropout : 0.2
regularisation : 0.001
learning_rate : 0.0001


In [35]:
from sklearn.metrics import f1_score

meilleur_reseau = tuner.get_best_models(num_models=1)[0]

probabilites = meilleur_reseau.predict(
    X_validation,
    verbose=0
).ravel()

predictions = (probabilites >= 0.5).astype(int)

f1_keras_tuner = f1_score(
    y_validation,
    predictions
)

print(f"F1 KerasTuner : {f1_keras_tuner:.4f}")

F1 KerasTuner : 0.8832


C:\Users\Utilisateur\Documents\formation\cicd_deep_learning\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


Une recherche aléatoire de 20 essais a été effectuée avec KerasTuner sur cinq hyperparamètres du réseau de neurones. Le modèle sélectionné obtient un F1-score de 0,8832 sur l'ensemble de validation interne.

Ce résultat est encourageant par rapport au réseau initial, mais ne suffit pas à démontrer une amélioration généralisable, puisque les hyperparamètres ont été sélectionnés à partir de cette même validation. La comparaison finale sera effectuée ultérieurement selon le protocole prévu.

## 6. Recherche des réglages avec Optuna

In [36]:
import optuna

In [37]:
def objectif(trial):

    # Les 5 hyperparamètres
    nb_couches = trial.suggest_categorical(
        "nb_couches", [1, 2, 3]
    )

    nb_neurones = trial.suggest_categorical(
        "nb_neurones", [16, 32, 64, 128]
    )

    dropout = trial.suggest_categorical(
        "dropout", [0.1, 0.2, 0.3, 0.4]
    )

    regularisation = trial.suggest_categorical(
        "regularisation", [0.0001, 0.001, 0.01]
    )

    learning_rate = trial.suggest_categorical(
        "learning_rate", [0.0001, 0.001, 0.01]
    )

    # Reproductibilité et nettoyage
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)

    # Réutilisation de la fonction de l'étape 5
    modele = construire_reseau(
        nb_couches,
        nb_neurones,
        dropout,
        regularisation,
        learning_rate
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    historique = modele.fit(
        X_tuning,
        y_tuning,
        validation_data=(X_validation, y_validation),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    # Même critère que KerasTuner : val_accuracy
    return max(historique.history["val_accuracy"])

In [38]:
sampler = optuna.samplers.RandomSampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

study.optimize(
    objectif,
    n_trials=20
)

[I 2026-10-23 16:08:52,087] A new study created in memory with name: no-name-64862037-90dd-4312-adaf-7ecfb7db3ea8
[I 2026-10-23 16:08:54,567] Trial 0 finished with value: 0.8865247964859009 and parameters: {'nb_couches': 2, 'nb_neurones': 16, 'dropout': 0.1, 'regularisation': 0.0001, 'learning_rate': 0.01}. Best is trial 0 with value: 0.8865247964859009.
[I 2026-10-23 16:08:56,308] Trial 1 finished with value: 0.8368794322013855 and parameters: {'nb_couches': 1, 'nb_neurones': 16, 'dropout': 0.2, 'regularisation': 0.01, 'learning_rate': 0.01}. Best is trial 0 with value: 0.8865247964859009.
[I 2026-10-23 16:08:58,289] Trial 2 finished with value: 0.8617021441459656 and parameters: {'nb_couches': 1, 'nb_neurones': 32, 'dropout': 0.3, 'regularisation': 0.0001, 'learning_rate': 0.01}. Best is trial 0 with value: 0.8865247964859009.
[I 2026-10-23 16:09:00,393] Trial 3 finished with value: 0.8829787373542786 and parameters: {'nb_couches': 2, 'nb_neurones': 32, 'dropout': 0.3, 'regularisatio

In [39]:
print("Meilleur score Optuna :", round(study.best_value, 4))

print("\nMeilleurs hyperparamètres :")

for nom, valeur in study.best_params.items():
    print(f"{nom} : {valeur}")

Meilleur score Optuna : 0.8972

Meilleurs hyperparamètres :
nb_couches : 3
nb_neurones : 16
dropout : 0.3
regularisation : 0.0001
learning_rate : 0.0001


In [40]:
score_keras = tuner.oracle.get_best_trials(1)[0].score
score_optuna = study.best_value

print(f"KerasTuner : {score_keras:.4f}")
print(f"Optuna     : {score_optuna:.4f}")

ecart = abs(score_keras - score_optuna)

print(f"Écart      : {ecart:.4f}")

KerasTuner : 0.8865
Optuna     : 0.8972
Écart      : 0.0106


In [41]:
# Reconstruire le meilleur modèle Optuna
keras.backend.clear_session()
keras.utils.set_random_seed(42)

meilleur_optuna = construire_reseau(
    **study.best_params
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

meilleur_optuna.fit(
    X_tuning,
    y_tuning,
    validation_data=(X_validation, y_validation),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

# Prédictions Optuna
probabilites = meilleur_optuna.predict(
    X_validation,
    verbose=0
).ravel()

predictions = (probabilites >= 0.5).astype(int)

f1_optuna = f1_score(y_validation, predictions)

# Comparaison
n = len(y_validation)

bruit_keras = 1.96 * np.sqrt(
    f1_keras_tuner * (1 - f1_keras_tuner) / n
)

bruit_optuna = 1.96 * np.sqrt(
    f1_optuna * (1 - f1_optuna) / n
)

marge = max(bruit_keras, bruit_optuna)
ecart = abs(f1_keras_tuner - f1_optuna)

print(f"F1 KerasTuner : {f1_keras_tuner:.4f}")
print(f"F1 Optuna     : {f1_optuna:.4f}")
print(f"Écart F1      : {ecart:.4f}")
print(f"Marge de bruit : ±{marge:.4f}")

if ecart <= marge:
    print("L'écart est dans la marge de bruit.")
else:
    print("L'écart dépasse la marge de bruit.")

F1 KerasTuner : 0.8832
F1 Optuna     : 0.8945
Écart F1      : 0.0113
Marge de bruit : ±0.0375
L'écart est dans la marge de bruit.


Une recherche d'hyperparamètres a été réalisée avec Optuna sur 20 essais, en utilisant le même espace de recherche et les mêmes conditions d'entraînement que KerasTuner.

Optuna obtient le meilleur score d'accuracy de validation (0,8972 contre 0,8865 pour KerasTuner). La comparaison des F1-scores donne également un avantage à Optuna (0,8945 contre 0,8832).

Cependant, l'écart de F1 (0,0113) reste inférieur à la marge de bruit indicative (0,0375). Il n'est donc pas possible de conclure à une supériorité significative d'Optuna selon le critère retenu.

Les ensembles de test final et de production n'ont pas été utilisés durant cette recherche

## 7. Verdict sur le test

In [42]:
meilleurs_reglages = study.best_params.copy()

print("Configuration retenue :")

for nom, valeur in meilleurs_reglages.items():
    print(f"{nom} : {valeur}")

Configuration retenue :
nb_couches : 3
nb_neurones : 16
dropout : 0.3
regularisation : 0.0001
learning_rate : 0.0001


In [43]:
graines = [42, 21, 123, 456, 789]

resultats_finaux = []

for graine in graines:

    print(f"\n===== Graine {graine} =====")

    # Découpage interne de l'apprentissage
    train_interne, val_interne = train_test_split(
        df_train,
        test_size=0.2,
        random_state=graine,
        stratify=df_train["cible"]
    )

    # TF-IDF appris uniquement sur l'entraînement interne
    vectoriseur = TfidfVectorizer(max_features=5000)

    X_tr = vectoriseur.fit_transform(train_interne["resume"])
    X_val = vectoriseur.transform(val_interne["resume"])

    y_tr = train_interne["cible"].to_numpy()
    y_val = val_interne["cible"].to_numpy()

    # 1. Naive Bayes
    nb = MultinomialNB()
    nb.fit(X_tr, y_tr)

    pred_nb = nb.predict(X_val)
    f1_nb = f1_score(y_val, pred_nb)

    # 2. Réseau initial
    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    reseau_initial = creer_modele(X_tr.shape[1])

    early_stop_initial = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    reseau_initial.fit(
        X_tr.toarray(),
        y_tr,
        validation_data=(X_val.toarray(), y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop_initial],
        verbose=0
    )

    proba_initial = reseau_initial.predict(
        X_val.toarray(),
        verbose=0
    ).ravel()

    pred_initial = (proba_initial >= 0.5).astype(int)

    f1_initial = f1_score(y_val, pred_initial)

    # 3. Réseau optimisé
    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    reseau_optimise = construire_reseau(
        **meilleurs_reglages
    )

    early_stop_optimise = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    reseau_optimise.fit(
        X_tr.toarray(),
        y_tr,
        validation_data=(X_val.toarray(), y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop_optimise],
        verbose=0
    )

    proba_optimise = reseau_optimise.predict(
        X_val.toarray(),
        verbose=0
    ).ravel()

    pred_optimise = (proba_optimise >= 0.5).astype(int)

    f1_optimise = f1_score(y_val, pred_optimise)

    resultats_finaux.append({
        "graine": graine,
        "Naive Bayes": f1_nb,
        "Réseau initial": f1_initial,
        "Réseau optimisé": f1_optimise
    })

    print(f"Naive Bayes : {f1_nb:.4f}")
    print(f"Réseau initial : {f1_initial:.4f}")
    print(f"Réseau optimisé : {f1_optimise:.4f}")


===== Graine 42 =====
Naive Bayes : 0.8736
Réseau initial : 0.8647
Réseau optimisé : 0.8945

===== Graine 21 =====
Naive Bayes : 0.8582
Réseau initial : 0.8612
Réseau optimisé : 0.8520

===== Graine 123 =====
Naive Bayes : 0.8507
Réseau initial : 0.8166
Réseau optimisé : 0.8298

===== Graine 456 =====
Naive Bayes : 0.8736
Réseau initial : 0.8511
Réseau optimisé : 0.8369

===== Graine 789 =====
Naive Bayes : 0.8529
Réseau initial : 0.8235
Réseau optimisé : 0.8529


In [44]:
df_finaux = pd.DataFrame(resultats_finaux)

display(df_finaux.round(4))

comparaison_finale = df_finaux[
    ["Naive Bayes", "Réseau initial", "Réseau optimisé"]
].agg(["mean", "std"])

display(comparaison_finale.round(4))

,graine,Naive Bayes,Réseau initial,Réseau optimisé
0,42,0.8736,0.8647,0.8945
1,21,0.8582,0.8612,0.8520
2,123,0.8507,0.8166,0.8298
3,456,0.8736,0.8511,0.8369
4,789,0.8529,0.8235,0.8529


,Naive Bayes,Réseau initial,Réseau optimisé
mean,0.8618,0.8434,0.8532
std,0.0111,0.0220,0.0251


In [45]:
moyennes = df_finaux[
    ["Naive Bayes", "Réseau initial", "Réseau optimisé"]
].mean()

n = len(val_interne)

marges = {}

for nom, p in moyennes.items():
    marge = 1.96 * np.sqrt(p * (1 - p) / n)
    marges[nom] = marge

    print(f"{nom} : F1 = {p:.4f} | Bruit = ±{marge:.4f}")

print("\n--- Comparaison avec le réseau optimisé ---")

for nom in ["Naive Bayes", "Réseau initial"]:

    ecart = moyennes["Réseau optimisé"] - moyennes[nom]
    marge = max(marges["Réseau optimisé"], marges[nom])

    print(f"\nRéseau optimisé vs {nom}")
    print(f"Écart F1 : {ecart:+.4f}")
    print(f"Marge : {marge:.4f}")

    if ecart > marge:
        print("Avantage supérieur à la marge de bruit")
    elif ecart < -marge:
        print("Réseau optimisé inférieur au-delà de la marge")
    else:
        print("Écart dans la marge de bruit")

Naive Bayes : F1 = 0.8618 | Bruit = ±0.0403
Réseau initial : F1 = 0.8434 | Bruit = ±0.0424
Réseau optimisé : F1 = 0.8532 | Bruit = ±0.0413

--- Comparaison avec le réseau optimisé ---

Réseau optimisé vs Naive Bayes
Écart F1 : -0.0086
Marge : 0.0413
Écart dans la marge de bruit

Réseau optimisé vs Réseau initial
Écart F1 : +0.0098
Marge : 0.0424
Écart dans la marge de bruit


In [46]:
graines = [42, 21, 123, 456, 789]

meilleures_epochs = []

for graine in graines:

    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    modele = construire_reseau(**meilleurs_reglages)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    historique = modele.fit(
        X_tuning,
        y_tuning,
        validation_data=(X_validation, y_validation),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    meilleure_epoch = np.argmin(
        historique.history["val_loss"]
    ) + 1

    meilleures_epochs.append(meilleure_epoch)

    print(f"Graine {graine} : meilleure epoch = {meilleure_epoch}")

Graine 42 : meilleure epoch = 58
Graine 21 : meilleure epoch = 60
Graine 123 : meilleure epoch = 55
Graine 456 : meilleure epoch = 56
Graine 789 : meilleure epoch = 53


In [47]:
epochs_finaux = int(np.median(meilleures_epochs))

print("Meilleures epochs :", meilleures_epochs)
print("Nombre d'epochs retenu :", epochs_finaux)

Meilleures epochs : [np.int64(58), np.int64(60), np.int64(55), np.int64(56), np.int64(53)]
Nombre d'epochs retenu : 56


In [48]:
tfidf_final = TfidfVectorizer(max_features=5000)

X_train_final = tfidf_final.fit_transform(
    df_train["resume"]
).toarray()

y_train_final = df_train["cible"].to_numpy()

print("Dimensions :", X_train_final.shape)
print("Epochs retenues :", epochs_finaux)

Dimensions : (1407, 5000)
Epochs retenues : 56


In [49]:
modeles_finaux = {}

for graine in graines:

    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    modele = construire_reseau(**meilleurs_reglages)

    modele.fit(
        X_train_final,
        y_train_final,
        epochs=epochs_finaux,
        batch_size=32,
        verbose=0
    )

    modeles_finaux[graine] = modele

    print(f"Modèle entraîné avec la graine {graine}")

Modèle entraîné avec la graine 42
Modèle entraîné avec la graine 21
Modèle entraîné avec la graine 123
Modèle entraîné avec la graine 456
Modèle entraîné avec la graine 789


In [50]:
graine_finale = 42

modele_final = modeles_finaux[graine_finale]

print("Modèle final retenu : graine", graine_finale)
print("Nombre d'epochs :", epochs_finaux)

Modèle final retenu : graine 42
Nombre d'epochs : 56


In [51]:
# Transformation du test avec le TF-IDF final
X_test_final = tfidf_final.transform(
    df_test["resume"]
).toarray()

y_test_final = df_test["cible"].to_numpy()

print("Dimensions du test :", X_test_final.shape)

Dimensions du test : (469, 5000)


In [52]:
# Prédictions du modèle final
probabilites_test = modele_final.predict(
    X_test_final,
    verbose=0
).ravel()

predictions_test = (probabilites_test >= 0.5).astype(int)

# Calcul du F1
f1_test = f1_score(
    y_test_final,
    predictions_test
)

print(f"F1 final sur le test : {f1_test:.4f}")

F1 final sur le test : 0.8425


In [53]:
# Nombre de films dans le test
n_test = len(y_test_final)

# Marge de bruit indicative
bruit_test = 1.96 * np.sqrt(
    f1_test * (1 - f1_test) / n_test
)

print(f"F1 final : {f1_test:.4f}")
print(f"Marge de bruit : ±{bruit_test:.4f}")
print(f"Nombre de films : {n_test}")

F1 final : 0.8425
Marge de bruit : ±0.0330
Nombre de films : 469


In [54]:
bilan = pd.DataFrame({
    "Modèle": [
        "Naive Bayes",
        "Réseau initial",
        "Réseau optimisé",
        "Réseau optimisé (test final)"
    ],
    "F1": [
        0.8618,
        0.8434,
        0.8522,
        f1_test
    ],
    "Évaluation": [
        "Validation interne",
        "Validation interne",
        "Validation interne",
        "Test final"
    ]
})

display(bilan.round(4))

,Modèle,F1,Évaluation
0,Naive Bayes,0.8618,Validation interne
1,Réseau initial,0.8434,Validation interne
2,Réseau optimisé,0.8522,Validation interne
3,Réseau optimisé (test final),0.8425,Test final


## Conclusion - Verdict sur le test

Le réseau optimisé avec Optuna a été comparé au réseau initial et au modèle Naive Bayes sur cinq graines différentes, à l'aide de validations internes.

Naive Bayes obtient le meilleur F1 moyen (0,8618), devant le réseau optimisé (0,8522) et le réseau initial (0,8434).

Toutefois, les différences observées restent inférieures aux marges de bruit indicatives. L'optimisation n'a donc pas permis de démontrer une amélioration claire des performances.

Le nombre d'epochs final a été fixé à 56, à partir de la médiane des meilleures epochs observées sur cinq entraînements. Le réseau optimisé a ensuite été réentraîné sur les 1 407 films de l'ensemble d'apprentissage, avec cinq graines différentes.

Le modèle retenu, correspondant à la graine 42, obtient un F1-score de 0,8425 sur les 469 films du test final, avec une marge de bruit indicative de ±0,0330.

En conclusion, le réseau optimisé présente des performances satisfaisantes, mais les résultats ne permettent pas d'affirmer qu'il surpasse clairement le réseau initial ou Naive Bayes selon le critère de marge de bruit retenu.

## 8. Sauvegarde

In [64]:
import json
import joblib

from pathlib import Path

In [65]:
# Dossier de sauvegarde
dossier_modeles = Path("../model")

# Création du dossier
dossier_modeles.mkdir(parents=True, exist_ok=True)

print("Dossier :", dossier_modeles.resolve())

Dossier : C:\Users\Utilisateur\Documents\formation\cicd_deep_learning\model


In [66]:
# Sauvegarde du réseau de neurones
modele_final.save(dossier_modeles / "model.keras")

print("Modèle Keras sauvegardé.")

Modèle Keras sauvegardé.


In [67]:
# Sauvegarde du TF-IDF
joblib.dump(
    tfidf_final,
    dossier_modeles / "vectoriseur.joblib"
)

print("Vectoriseur sauvegardé : vectoriseur.joblib")

Vectoriseur sauvegardé : vectoriseur.joblib


In [70]:
# Paramètres et métriques du modèle
informations = {
    "modele": "Reseau de neurones optimise Optuna",
    "graine": int(graine_finale),
    "epochs": int(epochs_finaux),
    "hyperparametres": {
        cle: float(valeur) if isinstance(valeur, float) else int(valeur)
        for cle, valeur in meilleurs_reglages.items()
    },
    "vectorisation": {
        "methode": "TF-IDF",
        "max_features": 5000,
        "vocabulaire": len(tfidf_final.vocabulary_)
    },
    "donnees": {
        "apprentissage": len(df_train),
        "test": len(df_test),
        "production": len(df_prod)
    },
    "metriques": {
        "f1_validation_naive_bayes": float(moyennes["Naive Bayes"]),
        "f1_validation_reseau_initial": float(moyennes["Réseau initial"]),
        "f1_validation_reseau_optimise": float(moyennes["Réseau optimisé"]),
        "f1_test_final": float(f1_test),
        "marge_bruit_test": float(bruit_test)
    },
    "seuil_classification": 0.5
}

# Sauvegarde JSON
with open(
    dossier_modeles / "metrics.json",
    "w",
    encoding="utf-8"
) as fichier:
    json.dump(informations, fichier, indent=4, ensure_ascii=False)

print("Paramètres et métriques sauvegardés.")

Paramètres et métriques sauvegardés.


In [71]:
# Échantillon de référence
reference = df_train[
    ["titre", "resume", "cible"]
].sample(n=10, random_state=42)

# Sauvegarde CSV
reference.to_csv(
    dossier_modeles / "reference.csv",
    index=False,
    encoding="utf-8"
)

print("Échantillon sauvegardé : reference.csv")

display(reference)

Échantillon sauvegardé : reference.csv


,titre,resume,cible
851,Things from Another World,Maroso Golfetto is a firm manager from Veneto ...,0
1660,Le pèril jeune,"Ten years after their Upper Sixth, Bruno, Momo...",0
663,AM1200,Cocky investment analyst Sam Larson has a lot ...,1
412,Gayby,Jenn and Matt are best friends from college wh...,0
1790,Mimic 2,When a cockroach-spread plague threatened to d...,1
1821,The Hills Run Red,"Tyler is obsessed by the horror movie ""The Hil...",1
450,The Curse of the Were-Rabbit,It's 'vege-mania' in Wallace and Gromit's neig...,0
849,House of Dracula,Dracula arrives at Dr. Edelman's office asking...,1
576,Shut In,Anna suffers from agoraphobia so crippling tha...,1
2141,Borderland,"Three college students, Phil, Ed, and Henry ta...",1


In [72]:
# Rechargement du modèle
modele_charge = keras.models.load_model(
    dossier_modeles / "model.keras",
    compile=False
)

# Rechargement du vectoriseur
vectoriseur_charge = joblib.load(
    dossier_modeles / "vectoriseur.joblib"
)

# Exemple de résumé
resume_exemple = [
    "A group of friends discovers a haunted house in the forest."
]

# Vectorisation
X_exemple = vectoriseur_charge.transform(
    resume_exemple
).toarray()

# Prédiction
probabilite = modele_charge.predict(
    X_exemple,
    verbose=0
)[0][0]

prediction = int(probabilite >= 0.5)

print("Probabilité classe 1 :", round(float(probabilite), 4))
print("Classe prédite :", prediction)

Probabilité classe 1 : 0.9934
Classe prédite : 1


In [73]:
# Fichiers attendus
fichiers_attendus = [
    "model.keras",
    "vectoriseur.joblib",
    "metrics.json",
    "reference.csv"
]

# Vérification
for nom in fichiers_attendus:
    chemin = dossier_modeles / nom

    if chemin.exists():
        taille = chemin.stat().st_size / 1024
        print(f"OK - {nom} ({taille:.2f} Ko)")
    else:
        print(f"MANQUANT - {nom}")

OK - model.keras (980.83 Ko)
OK - vectoriseur.joblib (178.44 Ko)
OK - metrics.json (0.82 Ko)
OK - reference.csv (5.76 Ko)


Le réseau de neurones optimisé avec Optuna a été sauvegardé au format model.keras, accompagné de son vectoriseur TF-IDF (vectoriseur.joblib).

Un fichier metrics.json conserve la version du modèle, le F1-score final de 0,8425 obtenu sur 469 films de test, les hyperparamètres et les versions des bibliothèques utilisées.

Un échantillon de dix films issus de l'ensemble d'apprentissage a également été enregistré dans reference.csv.

Le rechargement des artefacts a été vérifié afin de préparer leur intégration dans une API FastAPI.